# Automatic Urdu lexical Paraphrase Generation Model

This notebook implements a Seq2Seq networking using pure Keras.

In [0]:
# Start by importing all the things we'll need.
%tensorflow_version 1.x magic
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Dense, Embedding, CuDNNLSTM, Flatten, TimeDistributed, Dropout, LSTMCell, RNN
from tensorflow.keras.callbacks import ModelCheckpoint
from tensorflow.python.keras.utils import tf_utils
from tensorflow.keras import backend as K

%matplotlib notebook
import matplotlib.pyplot as plt
plt.ioff()

import unicodedata
import re
import numpy as np
import os
import time
import shutil

## Pulling and Pre-processing the Data


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Go to this URL in a browser: https://accounts.google.com/o/oauth2/auth?client_id=947318989803-6bn6qk8qdgf4n4g3pfee6491hc0brc4i.apps.googleusercontent.com&redirect_uri=urn%3aietf%3awg%3aoauth%3a2.0%3aoob&response_type=code&scope=email%20https%3a%2f%2fwww.googleapis.com%2fauth%2fdocs.test%20https%3a%2f%2fwww.googleapis.com%2fauth%2fdrive%20https%3a%2f%2fwww.googleapis.com%2fauth%2fdrive.photos.readonly%20https%3a%2f%2fwww.googleapis.com%2fauth%2fpeopleapi.readonly

Enter your authorization code:
··········
Mounted at /content/drive


In [0]:
class LanguageIndex():
    def __init__(self, lang):
        self.lang = lang
        self.word2idx = {}
        self.idx2word = {}
        self.vocab = set()
        self.create_index()
    def create_index(self):
        for phrase in self.lang:
            self.vocab.update(phrase.split(' '))
        self.vocab = sorted(self.vocab)
        self.word2idx["<pad>"] = 0
        self.idx2word[0] = "<pad>"
        for i,word in enumerate(self.vocab):
            self.word2idx[word] = i + 1
            self.idx2word[i+1] = word

# Download the file

path_to_file = '/content/drive/My Drive/PPDB lexical training/train_data/ppdb_lexical_train_data.csv'

def unicode_to_ascii(s):
    return ''.join(c for c in unicodedata.normalize('NFD', s) if unicodedata.category(c) != 'Mn')
def preprocess_sentence(w):
    w = unicode_to_ascii(w.lower().strip())
#     w = re.sub(r"([?.!,¿])", r" \1 ", w)
#     w = re.sub(r'[" "]+', " ", w)
#     w = re.sub(r"[^a-zA-Z?.!,¿]+", " ", w)
    w = w.rstrip().strip()
    w = "<start> " + w + " <end>"
    return w

def max_length(t):
    return max(len(i) for i in t)
   

def create_dataset(path):
    lines = open(path, encoding="UTF-8").read().strip().split("\n")
  
    word_pairs = [[preprocess_sentence(w) for w in l.split(",")] for l in lines]
    
    return word_pairs

def load_dataset(path):
    pairs = create_dataset(path)
    out_lang = LanguageIndex(sp for en, sp in pairs)
    in_lang = LanguageIndex(en for en, sp in pairs)
    input_data = [[in_lang.word2idx[s] for s in en.split(' ')] for en, sp in pairs]
    output_data = [[out_lang.word2idx[s] for s in sp.split(' ')] for en, sp in pairs]

    max_length_in, max_length_out = max_length(input_data), max_length(output_data)
    input_data = keras.preprocessing.sequence.pad_sequences(input_data, maxlen=max_length_in, padding="post")
    output_data = keras.preprocessing.sequence.pad_sequences(output_data, maxlen=max_length_out, padding="post")  
    return input_data, output_data, in_lang, out_lang, max_length_in, max_length_out

In our implementation we will need to provide two tensors for the target language. One is the "teacher tensor", which forces the decoder to follow a correct output stream. The other is the true "target tensor", which defines what the decoder should output given the teacher tensor. 

The only difference between the two is that the target tensor is just the teacher tensor shifted left by one word (and missing the "<START>" word).
    
To allow these two tensors to work in tandem, we will create two separate numpy arrays, even though the data is pretty much identical.

In [0]:
input_data, teacher_data, input_lang, target_lang, len_input, len_target = load_dataset(path_to_file)

target_data = [[teacher_data[n][i+1] for i in range(len(teacher_data[n])-1)] for n in range(len(teacher_data))]
target_data = keras.preprocessing.sequence.pad_sequences(target_data, maxlen=len_target, padding="post")
target_data = target_data.reshape((target_data.shape[0], target_data.shape[1], 1))

In [0]:
num_examples = 200000
input_data=input_data[:num_examples]
teacher_data = teacher_data[:num_examples]
target_data = target_data[:num_examples]

In [0]:
# Shuffle all of the data in unison. This training set has the longest (e.g. most complicated) data at the end,
# so a simple Keras validation split will be problematic if not shuffled.
p = np.random.permutation(len(input_data))
input_data = input_data[p]
teacher_data = teacher_data[p]
target_data = target_data[p]

BUFFER_SIZE = len(input_data)
BATCH_SIZE = 64
embedding_dim = 256
units = 1024
vocab_in_size = len(input_lang.word2idx)
vocab_out_size = len(target_lang.word2idx)

## Creating the Seq2Seq Model
Creating a Keras model which will be used to implement the encoder and decoder. This will be done with the Keras Functional API.

 We create this first model to train both concurrently by hooking them together. Later on we will use the trained weights from this model to create two separate models for inference.

I will be using an LSTM as I believe the LSTM does a better job at this task.

CuDNNLSTM works faster than the normal LSTM


In [7]:
# Create the Encoder layers first.
encoder_inputs = Input(shape=(len_input,))
encoder_emb = Embedding(input_dim=vocab_in_size, output_dim=embedding_dim)
encoder_lstm = CuDNNLSTM(units=units, return_sequences=True, return_state=True)
encoder_outputs, state_h, state_c = encoder_lstm(encoder_emb(encoder_inputs))
encoder_states = [state_h, state_c]

# Now create the Decoder layers.
decoder_inputs = Input(shape=(None,))
decoder_emb = Embedding(input_dim=vocab_out_size, output_dim=embedding_dim)
decoder_lstm = CuDNNLSTM(units=units, return_sequences=True, return_state=True)
decoder_lstm_out, _, _ = decoder_lstm(decoder_emb(decoder_inputs), initial_state=encoder_states)
# Two dense layers added to this model to improve inference capabilities.
decoder_d1 = Dense(units, activation="relu")
decoder_d2 = Dense(vocab_out_size, activation="softmax")
# Drop-out is added in the dense layers to help mitigate overfitting in this part of the model. Astute developers
# may want to add the same mechanism inside the LSTMs.
decoder_out = decoder_d2(Dropout(rate=.4)(decoder_d1(Dropout(rate=.4)(decoder_lstm_out))))

# Finally, create a training model which combines the encoder and the decoder.
# Note that this model has three inputs:
#  encoder_inputs=[batch,encoded_words] from input language (English)
#  decoder_inputs=[batch,encoded_words] from output language (Spanish). This is the "teacher tensor".
#  decoder_out=[batch,encoded_words] from output language (Spanish). This is the "target tensor".
model = Model([encoder_inputs, decoder_inputs], decoder_out)
# We'll use sparse_categorical_crossentropy so we don't have to expand decoder_out into a massive one-hot array.
#  Adam is used because it's, well, the best.
model.compile(optimizer='adam', loss="sparse_categorical_crossentropy", metrics=['sparse_categorical_accuracy'])

Instructions for updating:
Call initializer instance with the dtype argument instead of passing it to the constructor
Instructions for updating:
If using Keras pass *_constraint arguments to layers.


In [8]:
model.summary()

Model: "model"
__________________________________________________________________________________________________
Layer (type)                    Output Shape         Param #     Connected to                     
input_1 (InputLayer)            [(None, 18)]         0                                            
__________________________________________________________________________________________________
input_2 (InputLayer)            [(None, None)]       0                                            
__________________________________________________________________________________________________
embedding (Embedding)           (None, 18, 256)      2262784     input_1[0][0]                    
__________________________________________________________________________________________________
embedding_1 (Embedding)         (None, None, 256)    2290944     input_2[0][0]                    
______________________________________________________________________________________________

In [0]:
filepath="/content/drive/My Drive/PPDB lexical training/saved_models/weights-improvement-{epoch:02d}-{sparse_categorical_accuracy:.2f}.hdf5"

# filepath="/content/drive/My Drive/PPDB phrasal training/saved_models/weights-improvement-{epoch:02d}-{sparse_categorical_accuracy:.2f}.hdf5"
checkpoint = tf.keras.callbacks.ModelCheckpoint(filepath, monitor='sparse_categorical_accuracy', verbose=1, save_best_only=True, mode='max')
callbacks_list = [checkpoint]

The next cell trains the model and plots the training and verification losses.

In [9]:
# Note, we use 10% of our data for validation.
epochs = 15
model.fit([input_data, teacher_data], target_data,
                 batch_size=BATCH_SIZE,
                 epochs=epochs,
                 validation_split=0.1,
                # callbacks=callbacks_list
          )

Instructions for updating:
Use tf.where in 2.0, which has the same broadcast rule as np.where
Train on 180000 samples, validate on 20000 samples
Epoch 1/15
180000/180000 [==============================] - 404s 2ms/sample - loss: 0.4851 - sparse_categorical_accuracy: 0.9294 - val_loss: 0.4878 - val_sparse_categorical_accuracy: 0.9319
Epoch 2/15
180000/180000 [==============================] - 398s 2ms/sample - loss: 0.4303 - sparse_categorical_accuracy: 0.9347 - val_loss: 0.4251 - val_sparse_categorical_accuracy: 0.9356
Epoch 3/15
180000/180000 [==============================] - 398s 2ms/sample - loss: 0.4289 - sparse_categorical_accuracy: 0.9349 - val_loss: 0.4139 - val_sparse_categorical_accuracy: 0.9360
Epoch 4/15
180000/180000 [==============================] - 398s 2ms/sample - loss: 0.4056 - sparse_categorical_accuracy: 0.9361 - val_loss: 0.4039 - val_sparse_categorical_accuracy: 0.9364
Epoch 5/15
180000/180000 [==============================] - 398s 2ms/sample - loss: 0.3957 - sp

In [0]:
# from tensorflow.keras.models import load_model

# # load model
# model = load_model('/content/drive/My Drive/PPDB phrasal training/saved_models/weights-improvement-01-0.00.hdf5')
# # summarize model.
# model.summary()

## Inference
The trick to our Keras seq2seq model is that Inference will require two separate models from training. This is because during training, we used a "teacher tensor" to force the model to continuously make predictions assuming it had gotten everything before right. Now we want it to make those predictions all by itself.

To accomplish this, we need to break up the encoder and decoder mechanisms. We will then run the entire input sequence through the encoder, then form the output by executing the decoder one step at a time.

In [0]:
# Create the encoder model from the tensors we previously declared.
encoder_model = Model(encoder_inputs, [encoder_outputs, state_h, state_c])

# Generate a new set of tensors for our new inference decoder. Note that we are using new tensors, 
# this does not preclude using the same underlying layers that we trained on. (e.g. weights/biases).
inf_decoder_inputs = Input(shape=(None,), name="inf_decoder_inputs")
# We'll need to force feed the two state variables into the decoder each step.
state_input_h = Input(shape=(units,), name="state_input_h")
state_input_c = Input(shape=(units,), name="state_input_c")
decoder_res, decoder_h, decoder_c = decoder_lstm(
    decoder_emb(inf_decoder_inputs), 
    initial_state=[state_input_h, state_input_c])
inf_decoder_out = decoder_d2(decoder_d1(decoder_res))
inf_model = Model(inputs=[inf_decoder_inputs, state_input_h, state_input_c], 
                  outputs=[inf_decoder_out, decoder_h, decoder_c])

Now that we've got our models, let's define some functions that will assist us in actually translating a given string of text.

In [0]:
# Converts the given sentence (just a string) into a vector of word IDs
# using the language specified. This can be used for either the input or target languages.
# Output is 1-D: [timesteps/words]
def sentence_to_vector(sentence, lang):
    pre = preprocess_sentence(sentence)
    vec = np.zeros(len_input)
    sentence_list = [lang.word2idx[s] for s in pre.split(' ')]
    for i,w in enumerate(sentence_list):
        vec[i] = w
    return vec

# Given an input string, an encoder model (infenc_model) and a decoder model (infmodel),return a paraphrased string.
def translate(input_sentence, infenc_model, infmodel, attention=False):
    sv = sentence_to_vector(input_sentence, input_lang)
    # Reshape so we can use the encoder model. New shape=[samples,sequence length]
    sv = sv.reshape(1,len(sv))
    [emb_out, sh, sc] = infenc_model.predict(x=sv)
    
    i = 0
    start_vec = target_lang.word2idx["<start>"]
    stop_vec = target_lang.word2idx["<end>"]
    # We will continuously feed cur_vec as an input into the decoder to produce the next word,
    # which will be assigned to cur_vec. Start it with "<start>".
    cur_vec = np.zeros((1,1))
    cur_vec[0,0] = start_vec
    cur_word = "<start>"
    output_sentence = ""
    # Start doing the feeding. Terminate when the model predicts an "<end>" or we reach the end
    # of the max target sentence length.
    while cur_word != "<end>" and i < (len_target-1):
        i += 1
        if cur_word != "<start>":
            output_sentence = output_sentence + " " + cur_word
        x_in = [cur_vec, sh, sc]
        # This will allow us to accomodate attention models, which we will talk about later.
        if attention:
            x_in += [emb_out]
        [nvec, sh, sc] = infmodel.predict(x=x_in)
        # The output of the model is a massive softmax vector with one spot for every possible word. Convert
        # it to a word ID using argmax().
        cur_vec[0,0] = np.argmax(nvec[0,0])
        cur_word = target_lang.idx2word[np.argmax(nvec[0,0])]
    return output_sentence
    

In [12]:
import pandas as pd
test_data = pd.read_excel('/content/drive/My Drive/PPDB lexical training/test_data/lexical_test_data.xlsx' , names=['a','b'])
source = test_data['a'].values
target = test_data['b'].values
n=1
a = []
b = []
c = []
for n in range(10300):
  pred = translate(source[n], encoder_model, inf_model)
  print(n)
  print("Source: " , source[n])
  print("pred:   ", pred)
  print("Target" , target[n])
  print('--------------------------------------------------------')
  a.append(source[n])
  b.append(pred)
  c.append(target[n])

0
Source:  ٹوٹاھوا
pred:     ختم ہوگیا
Target ختم
--------------------------------------------------------
1
Source:  سمیت
pred:     دوبارہ تعمیر
Target پر مشتمل ہے
--------------------------------------------------------
2
Source:  قائل
pred:     اضافہ
Target تسلیم کیا
--------------------------------------------------------
3
Source:  ڈیزائن کیا گیا
pred:     پیش گويی
Target ارادہ
--------------------------------------------------------
4
Source:  حل کرنا
pred:     جواب دیں
Target قانونی چارہ جوئی
--------------------------------------------------------
5
Source:  سمیت
pred:     دوبارہ تعمیر
Target پر مشتمل ہے
--------------------------------------------------------
6
Source:  پختہ
pred:     شراب
Target مرمت
--------------------------------------------------------
7
Source:  بنیادیں
pred:     اداروں
Target وجوہات
--------------------------------------------------------
8
Source:  خطاب کیا
pred:     ختم ہوگیا
Target فرض کیا
--------------------------------------------------------
9
So

In [0]:
df1= pd.DataFrame(a, columns= ['source']) 
df2= pd.DataFrame(b, columns= ['Predicted']) 
df3= pd.DataFrame(c, columns= ['target']) 

In [0]:
frames = [df1,df2,df3]
outputs = pd.concat(frames,axis=1)
outputs.to_csv('/content/drive/My Drive/PPDB lexical training/predictions_on_15_EPOCHS.csv', index=False)

In [15]:
outputs

,source,Predicted,target
0,ٹوٹاھوا,ختم ہوگیا,ختم
1,سمیت,دوبارہ تعمیر,پر مشتمل ہے
2,قائل,اضافہ,تسلیم کیا
3,ڈیزائن کیا گیا,پیش گويی,ارادہ
4,حل کرنا,جواب دیں,قانونی چارہ جوئی
...,...,...,...
10295,فارورڈز,بیان کرتا ہے,حصص
10296,یاد دلایا,یاد رکھنا,تجویز کردہ
10297,مسلط,دوبارہ تعمیر,فراہم کی
10298,تفتیش کی,دوبارہ تعمیر,روشن ضمیر
